# MorphIQ SDK for Python — Start Here

Price a **European call** and an **American put**, then try the async client.
These inputs are synthetic. No market-data subscription is needed.

This notebook makes real requests only when you execute the pricing cells.
Each sync cell makes one request; the async cell makes two more. Running all
cells therefore makes **four pricing requests**, which may consume your API
allowance. Opening Jupyter does not call the API.

Docker includes the required Python environment. For a downloaded notebook,
install this repository with its `notebooks` extra using the quickstart, then
open it in JupyterLab. The notebook is self-contained; it needs no examples imports.

## Feature notebooks

- [European pricing](pricing/european.ipynb)
- [American pricing](pricing/american.ipynb)
- [Async pricing](pricing/async.ipynb)


In [ ]:
import sys

import morphiq_sdk

print("Python:", sys.version.split()[0])
print("MorphIQ SDK:", morphiq_sdk.__version__)

## Connect to your API

Get your API key from the
[MorphIQ Labs developer portal](https://developers.morphiqlabs.com).
The SDK connects to `https://api.morphiqlabs.com` automatically. Before starting
Jupyter or Docker Compose, set your API key in the terminal:

```sh
export MORPHIQ_API_KEY="YOUR_API_KEY"
```

Docker Compose passes the key into the notebook container. If it is unset,
the cell below prompts. The key prompt is masked: **do not paste your key into a code
cell**. The key stays in this kernel's process environment, not notebook source
or outputs; clear it in the cleanup cell and shut down the kernel when finished.

For a local or controlled test service, set `MORPHIQ_API_BASE_URL` before startup
to override the default API URL.

Each pricing cell contains a complete example: request inputs, API call, response
fields and error handling. The code is copied from the executable Python scripts.
The server validates model domains and selects the pricing algorithm.


In [ ]:
import os
from getpass import getpass

if not os.environ.get("MORPHIQ_API_KEY"):
    os.environ["MORPHIQ_API_KEY"] = getpass("MorphIQ API key: ")

## 1. European call

Spot and strike are both 100 in the same currency. Expiry is **1 year**,
volatility is **0.20 = 20%**, interest is **0.05 = 5%**, and dividend yield is zero.
The Python default sends `pricing_model="auto"`; read the returned model and
algorithm version. The price uses the same currency units as spot and strike;
no contract multiplier is applied.


In [ ]:
import os
import sys
from getpass import getpass

from morphiq_sdk import APIError, MorphIQ, PriceRequest, ResponseError, TransportError

# Set MORPHIQ_API_KEY in your environment, or enter it at the masked prompt.
api_key = os.environ.get("MORPHIQ_API_KEY") or getpass("MorphIQ API key: ")

# Synthetic European call; spot and strike use the same currency.
request = PriceRequest(
    exercise_style="european",
    option_type="call",
    spot=100.0,
    strike=100.0,
    time_to_expiry_years=1.0,  # Years from valuation.
    risk_free_rate=0.05,  # 5% annual rate.
    dividend_yield=0.0,
    volatility=0.20,  # 20% annual volatility.
    pricing_model="auto",
)

try:
    # The client uses https://api.morphiqlabs.com and closes its connection pool here.
    with MorphIQ(api_key=api_key) as client:
        response = client.price(request)
except APIError as error:
    print(f"HTTP {error.status_code}: {error.code}: {error.message}", file=sys.stderr)
    print(f"Request ID: {error.request_id}", file=sys.stderr)
    charged = error.metadata.fcu_charged if error.metadata else None
    print(
        f"Commercial CU charged: {charged if charged is not None else 'not reported'}",
        file=sys.stderr,
    )
    raise SystemExit(1) from None
except TransportError:
    # Includes timeouts. The service may already have committed a charge.
    raise SystemExit(
        "No complete response; execution and charges may be unknown. No retry."
    ) from None
except ResponseError as error:
    raise SystemExit(
        f"Invalid response; request_id={error.request_id}. Charges may be unknown. No retry."
    ) from None

print(f"Price: {response.result.price}")
print(f"Model: {response.result.pricing_model} ({response.result.model_version})")
print(f"Request ID: {response.metadata.request_id}")
for engine in response.metadata.engines:
    print(f"Engine: {engine.name} {engine.version} ({engine.revision})")
# Missing commercial CU is unreported, not zero.
charged = response.metadata.fcu_charged
print("Commercial CU charged:", charged if charged is not None else "not reported")

## 2. American put

Spot is 100, strike is 105, expiry is **0.5 years**, volatility is 20%, interest
is 5%, and the continuous dividend yield is **0.01 = 1%**. The price uses the same
currency units as spot and strike; no contract multiplier is applied.
Cash-dividend schedules and date/day-count conversion are outside this first example.


In [ ]:
import os
import sys
from getpass import getpass

from morphiq_sdk import APIError, MorphIQ, PriceRequest, ResponseError, TransportError

# Set MORPHIQ_API_KEY in your environment, or enter it at the masked prompt.
api_key = os.environ.get("MORPHIQ_API_KEY") or getpass("MorphIQ API key: ")

# Synthetic American put; spot and strike use the same currency.
request = PriceRequest(
    exercise_style="american",
    option_type="put",
    spot=100.0,
    strike=105.0,
    time_to_expiry_years=0.5,  # Years from valuation.
    risk_free_rate=0.05,  # 5% annual rate.
    dividend_yield=0.01,
    volatility=0.20,  # 20% annual volatility.
    pricing_model="auto",
)

try:
    # The client uses https://api.morphiqlabs.com and closes its connection pool here.
    with MorphIQ(api_key=api_key) as client:
        response = client.price(request)
except APIError as error:
    print(f"HTTP {error.status_code}: {error.code}: {error.message}", file=sys.stderr)
    print(f"Request ID: {error.request_id}", file=sys.stderr)
    charged = error.metadata.fcu_charged if error.metadata else None
    print(
        f"Commercial CU charged: {charged if charged is not None else 'not reported'}",
        file=sys.stderr,
    )
    raise SystemExit(1) from None
except TransportError:
    # Includes timeouts. The service may already have committed a charge.
    raise SystemExit(
        "No complete response; execution and charges may be unknown. No retry."
    ) from None
except ResponseError as error:
    raise SystemExit(
        f"Invalid response; request_id={error.request_id}. Charges may be unknown. No retry."
    ) from None

print(f"Price: {response.result.price}")
print(f"Model: {response.result.pricing_model} ({response.result.model_version})")
print(f"Request ID: {response.metadata.request_id}")
for engine in response.metadata.engines:
    print(f"Engine: {engine.name} {engine.version} ({engine.revision})")
# Missing commercial CU is unreported, not zero.
charged = response.metadata.fcu_charged
print("Commercial CU charged:", charged if charged is not None else "not reported")

## 3. The same requests with async

Jupyter supports top-level `await`; do not use `asyncio.run` in a running notebook
event loop. These requests are deliberately sequential, allowing use with plans
that permit only one request at a time. This cell makes **two pricing requests**.


In [ ]:
import os
import sys
from getpass import getpass

from morphiq_sdk import APIError, AsyncMorphIQ, PriceRequest, ResponseError, TransportError

# Set MORPHIQ_API_KEY in your environment, or enter it at the masked prompt.
api_key = os.environ.get("MORPHIQ_API_KEY") or getpass("MorphIQ API key: ")

# Package each option as its own request; this is two sequential API calls.
requests = [
    PriceRequest(
        exercise_style="european",
        option_type="call",
        spot=100.0,
        strike=100.0,
        time_to_expiry_years=1.0,  # Years from valuation.
        risk_free_rate=0.05,  # 5% annual rate.
        dividend_yield=0.0,
        volatility=0.20,  # 20% annual volatility.
        pricing_model="auto",
    ),
    PriceRequest(
        exercise_style="american",
        option_type="put",
        spot=100.0,
        strike=105.0,
        time_to_expiry_years=0.5,  # Years from valuation.
        risk_free_rate=0.05,  # 5% annual rate.
        dividend_yield=0.01,
        volatility=0.20,  # 20% annual volatility.
        pricing_model="auto",
    ),
]

try:
    async with AsyncMorphIQ(api_key=api_key) as client:
        for request in requests:
            response = await client.price(request)
            print(f"{response.result.exercise_style} {response.result.option_type}")
            print(f"Price: {response.result.price}")
            print(f"Model: {response.result.pricing_model} ({response.result.model_version})")
            print(f"Request ID: {response.metadata.request_id}")
            for engine in response.metadata.engines:
                print(f"Engine: {engine.name} {engine.version} ({engine.revision})")
            # Missing commercial CU is unreported, not zero.
            charged = response.metadata.fcu_charged
            print("Commercial CU charged:", charged if charged is not None else "not reported")
except APIError as error:
    print(f"HTTP {error.status_code}: {error.code}: {error.message}", file=sys.stderr)
    print(f"Request ID: {error.request_id}", file=sys.stderr)
    charged = error.metadata.fcu_charged if error.metadata else None
    print(
        f"Commercial CU charged: {charged if charged is not None else 'not reported'}",
        file=sys.stderr,
    )
    raise SystemExit(1) from None
except TransportError:
    # Includes timeouts. The service may already have committed a charge.
    raise SystemExit(
        "No complete response; execution and charges may be unknown. No retry."
    ) from None
except ResponseError as error:
    raise SystemExit(
        f"Invalid response; request_id={error.request_id}. Charges may be unknown. No retry."
    ) from None

## Reading results and failures

Use `response.result.price`, `response.result.pricing_model`, and
`response.metadata.request_id`. Commercial charges are `metadata.fcu_charged`;
if absent they are **unreported**, not zero. `units_charged` is a separate
internal compute weight. Keep the request ID when contacting support.

`APIError` exposes HTTP status, server code, message, request ID, optional charge
metadata and `Retry-After`. `RequestTimeout` means execution/charges may be
unknown. Both clients make **no automatic retries**, including after timeouts,
and do not follow redirects. Correct the input/key/endpoint or inspect the
reported charge before deliberately submitting another request.

## Finish

The next cell removes the API key from this kernel's environment. Shut down the
kernel to release its remaining memory. If you exported the key in your terminal,
run `unset MORPHIQ_API_KEY` there when finished; `docker compose down` removes the
container configured with that key. Clear outputs before sharing a notebook.


In [ ]:
if "MORPHIQ_API_KEY" in os.environ:
    del os.environ["MORPHIQ_API_KEY"]
print("API key removed from the kernel environment.")